# Field disease model — Colab GPU training

Trains the MobileNetV3-Large field model on GPU with more epochs / more data than the CPU run.
It reuses the repo's `ml/` scripts unchanged.

**First:** Runtime → Change runtime type → **T4 GPU**.

**Getting the code onto Colab:** easiest is to push this repo to GitHub once and set `REPO_URL` below.
No remote yet? On your machine:
```
gh repo create plant-monitoring-system --private --source . --push   # or add a remote + git push
```
Fallback: zip your `ml/` + `edge-server/app/models/` folders and upload via the Files panel, then skip the clone cell.

In [ ]:
!nvidia-smi -L   # confirm a GPU is attached

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/<you>/plant-monitoring-system.git"
import os
if REPO_URL:
    !git clone -q "$REPO_URL" project
    %cd project
else:
    raise SystemExit("Set REPO_URL, OR upload ml/ + edge-server/app/models/ to /content and skip this cell.")

In [ ]:
!pip install -q -r ml/requirements.txt

## Data
PlantDoc = real field images (our test set is field-only, so accuracy stays honest).

In [ ]:
!git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset datasets/plantdoc
!python ml/plantdoc_to_pv.py --src datasets/plantdoc --out datasets/pv15

### (Optional but recommended) add PlantVillage for all 15 classes + more data
Restores Potato-healthy / Target-Spot / Spider-mites and adds thousands of images.
The lab images are degraded to field-like by `ml/augment.py`. Skip this cell to train PlantDoc-only (13 classes).

In [ ]:
!git clone -q --depth 1 https://github.com/spMohanty/PlantVillage-Dataset datasets/plantvillage
!python ml/add_plantvillage.py \
    --pv-src datasets/plantvillage/raw/color \
    --into datasets/pv15 \
    --labels edge-server/app/models/plantvillage_labels.json \
    --per-class 400

## Baseline — current model on the field test set (the number to beat)

In [ ]:
!python ml/evaluate.py \
    --model edge-server/app/models/plantvillage_resnet18_15cls.onnx \
    --labels edge-server/app/models/plantvillage_labels.json \
    --data datasets/pv15/test --img-size 128

## Train on GPU
200 epochs, batch 64 — or paste the winning `--backbone/--img-size/--lr/--weight-decay/--label-smoothing/--drop-rate` from the sweep above. To init from PDDD-PreTrain instead of ImageNet, upload the weights and add `--pretrained-ckpt weights/pddd_mnv3.pth`.

In [ ]:
!python ml/sweep.py --data datasets/pv15 --trials 30 --epochs 20 --batch 64 --workers 2 --out ml/runs/sweep

## Export FP32 + INT8, then evaluate on the field test set
`--quant` options: `static` (default; smallest, ~1-pt loss), `dynamic` (no calibration), `fp16` (near-lossless, half size), `none` (FP32 only). Add `--exclude-nodes <names>` to keep sensitive layers in FP32 (mixed precision) if static INT8 costs too much. On a Pi 5, FP32 is already fast — INT8 mainly saves size/memory.

In [ ]:
!python ml/train.py \
    --data datasets/pv15 \
    --epochs 200 --img-size 224 --batch 64 --workers 2 \
    --out ml/runs/colab

## Export FP32 + static INT8, then evaluate on the field test set

In [ ]:
!python ml/export_onnx.py \
    --ckpt ml/runs/colab/best.pt \
    --calib-dir datasets/pv15/val \
    --out ml/runs/colab/export

In [ ]:
import glob
fp32 = sorted(glob.glob('ml/runs/colab/export/field_mnv3_*cls.onnx'))
fp32 = [m for m in fp32 if 'int8' not in m][0]
int8 = fp32.replace('.onnx', '_int8.onnx')
labels = 'ml/runs/colab/export/field_labels.json'
print('=== new FP32 ===')
!python ml/evaluate.py --model "$fp32" --labels "$labels" --data datasets/pv15/test --img-size 224
print('=== new INT8 ===')
!python ml/evaluate.py --model "$int8" --labels "$labels" --data datasets/pv15/test --img-size 224

## Download the trained model back to your machine
Drop the `.onnx` + `field_labels.json` into `edge-server/app/models/`, set `MODEL_PATH` + the labels path in `edge-server/.env`, and restart. The server auto-detects the 224px input.

In [ ]:
from google.colab import files
import glob
for f in glob.glob('ml/runs/colab/export/*'):
    files.download(f)